# 04 · Evaluation

How well the pipeline finds contradictions that were deliberately injected into *The Wonderful Wizard of Oz* (`data/test_corpus/oz_injections.json`), measured as precision and recall. Three runs are compared:

| Run | Where | Model | Pipeline |
|---|---|---|---|
| `laptop_v1` | 8 GB MacBook Air | qwen3:4b-instruct | extraction v4, checker v2 (first version) |
| `kaggle_4b` | Kaggle T4 GPU | qwen3:4b-instruct | extraction v5, checker v3 (with double-check) |
| `kaggle_14b` | Kaggle T4 GPU | qwen3:14b | extraction v5, checker v3 (with double-check) |

The numbers come from `scripts/evaluate.py` (saved in `data/results/`); this notebook only displays them, so it needs nothing but `pandas`.

- **Recall:** the share of injected contradictions that were flagged.
- **Precision:** the share of flags that point at an injected contradiction.
- **F1:** recall and precision combined.
- **Tier agreement:** how often the flag's confidence matches the tier the injection was designed for.

## 0. Setup and the three runs side by side

In [ ]:
import importlib.util

# This notebook only reads saved results, so pandas is all it needs.
if importlib.util.find_spec("pandas") is None:
    %pip install --quiet pandas
    print("Installed pandas. Restart the kernel (Restart in the toolbar), then run the notebook again.")
else:
    print("pandas is installed.")

In [ ]:
import json
import pathlib

import pandas as pd

pd.set_option("display.max_colwidth", 140)
RESULTS = pathlib.Path("../data/results")
RUNS = {"laptop_v1": RESULTS, "kaggle_4b": RESULTS / "kaggle_4b", "kaggle_14b": RESULTS / "kaggle_14b"}


def load(run):
    return json.loads((RUNS[run] / "evaluation.json").read_text())


def f1(e):
    p, r = e["precision"] or 0.0, e["recall"]
    return 2 * p * r / (p + r) if p + r else 0.0


rows = {}
for run in RUNS:
    e = load(run)
    rows[run] = {
        "model": e["meta"]["model"],
        "pipeline": f"extraction {e['meta']['extraction']}, checker {e['meta']['checker']}",
        "recall": f"{e['detected']}/{e['contradictions']} ({e['recall']:.0%})",
        "precision": f"{e['true_positive_flags']}/{e['flags']} ({(e['precision'] or 0):.0%})",
        "F1": f"{f1(e):.0%}",
        "high-tier precision": f"{e['by_confidence']['high']['true_positives']:.0f}/{e['by_confidence']['high']['flags']:.0f}",
        "false alarms on original book": len(json.loads((RUNS[run] / "flags_oz.json").read_text())),
    }
pd.DataFrame(rows).T

In [ ]:
# Pick the run to look at in detail: "laptop_v1", "kaggle_4b" or "kaggle_14b".
RUN = "kaggle_14b"

result = load(RUN)
print(" · ".join(f"{k}: {v}" for k, v in result["meta"].items()))
pd.Series({
    "injected contradictions": result["contradictions"],
    "detected": result["detected"],
    "recall": f"{result['recall']:.0%}",
    "flags raised": result["flags"],
    "flags pointing at an injection": result["true_positive_flags"],
    "precision": "n/a" if result["precision"] is None else f"{result['precision']:.0%}",
    "F1": f"{f1(result):.0%}",
    "false alarms on control edits": result["control_false_alarms"],
    "tier agreement": "n/a" if result["tier_agreement"] is None else f"{result['tier_agreement']:.0%}",
}, name="value").to_frame()

## Recall by category

In [ ]:
pd.DataFrame(result["by_category"]).T

## Precision by confidence tier

High-confidence flags should be the most precise.

In [ ]:
pd.DataFrame(result["by_confidence"]).T

## Every injected contradiction

In [ ]:
pd.DataFrame(result["injections"])[
    ["id", "chapter", "category", "replace", "detected", "detected_in_chapter",
     "expected_confidence", "predicted_confidence", "reason"]
].set_index("id")

## Full reports

Each run's `evaluation_report.md` (in `data/results/`, `data/results/kaggle_4b/` and `data/results/kaggle_14b/`) has the same tables plus every false alarm with the checker's reason.